# Attention forward pass

In [2]:
import numpy as np


In [6]:
np.random.seed(42)
N = 5 # Len of sequence
D = 10 # Dimention of the embedding
d_k = 4 # Dimention of the internal state

X = np.random.rand(N, D)
Wq = np.random.rand(D, d_k)
Wk = np.random.rand(D, d_k)
Wv = np.random.rand(D, d_k)


q = X @ Wq # Result shape = (N, d_k). 
k = X @ Wk
v = X @ Wv

scores = (q @ k.T) # (N, N)
norm = scores / np.sqrt(d_k)

# Softmax
exp = np.exp(norm) # (N, N)
sum_exp = np.sum(exp, axis=1, keepdims=True) # (N, 1)
attention = exp / sum_exp # En cada fila hay las atenciones para una sola query

result = attention @ v # (N, N) @ (N, d_k) = (N, d_k)

loss = np.sum(result)
loss

print(f'Sum exp {np.sum(np.exp(scores), axis=1, keepdims=True)}')
print(f'Scores {scores}')
print(f'Attention: {attention}')
print(f'Sum {np.sum(attention, axis=1)}')
print(f'Result: {result}')

Sum exp [[1.81758557e+11]
 [4.99842208e+07]
 [1.30831869e+09]
 [1.16781266e+12]
 [5.84159344e+08]]
Scores [[25.77798284 18.64742887 19.48398366 23.86705031 21.05392398]
 [17.41027692 12.56780902 13.08852887 16.24935311 14.15475403]
 [20.86580444 15.14410588 15.57329611 18.67234283 16.64457634]
 [27.73448991 20.11888043 20.41837683 24.71968864 21.85938877]
 [20.05451185 14.56520542 14.72214283 17.89288909 15.923275  ]]
Attention: [[0.64510652 0.01824951 0.02772725 0.2481291  0.06078761]
 [0.51019242 0.04531119 0.0587866  0.28552399 0.1001858 ]
 [0.63160775 0.03614069 0.04479132 0.21093241 0.07652783]
 [0.75616668 0.01678452 0.01949592 0.16747954 0.04007334]
 [0.62505956 0.04017294 0.04345224 0.21209527 0.07921998]]
Sum [1. 1. 1. 1. 1.]
Result: [[2.8008452  2.24120063 1.59079464 2.44745014]
 [2.79069057 2.14378157 1.50042271 2.36748655]
 [2.7716833  2.23145586 1.5976439  2.43297517]
 [2.77443958 2.32262364 1.68618768 2.50579409]
 [2.77092348 2.2274314  1.5955526  2.42962055]]


In [11]:
# Partial derivatives for each opeartion
dL_dresult = np.ones_like(result) # Derivative of the loss with respect to the result
dL_dattention = dL_dresult @ v.T
dL_dv = attention.T @ dL_dattention
dL_dexp = dL_dattention @ attention
dL_dsum_exp = dL_dexp / sum_exp
dL_dnorm = dL_dsum_exp * exp
dL_dscores = dL_dnorm * np.exp(norm)  # Derivative of

# the softmax function
dL_dq = dL_dscores @ k
dL_dk = dL_dscores.T @ q
dL_dWq = X.T @ dL_dq
dL_dWk = X.T @ dL_dk
dL_dWv = X.T @ dL_dv

# Update weights
learning_rate = 0.01
Wq -= learning_rate * dL_dWq
Wk -= learning_rate * dL_dWk
Wv -= learning_rate * dL_dWv

# Compute loss after update
q = X @ Wq
k = X @ Wk
v = X @ Wv
scores = (q @ k.T)
norm = scores / np.sqrt(d_k)
exp = np.exp(norm)
sum_exp = np.sum(exp, axis=1, keepdims=True)
attention = exp / sum_exp
result = attention @ v
loss = np.sum(result)
print(f'Updated loss: {loss}')


ValueError: operands could not be broadcast together with shapes (10,4) (10,5) (10,4) 